# Hierarchical Clustering

## Assignment (b): Hierarchical Clustering using Scikit-learn

**Author:** Nitish  
**Date:** December 2024

---

## Table of Contents
1. [Introduction](#introduction)
2. [Theory and Concepts](#theory)
3. [Dataset Preparation](#dataset)
4. [Agglomerative Clustering](#agglomerative)
5. [Dendrogram Analysis](#dendrogram)
6. [Different Linkage Methods](#linkage)
7. [Clustering Quality Metrics](#metrics)
8. [Real-World Application](#realworld)
9. [Conclusion](#conclusion)

---

<a id='introduction'></a>
## 1. Introduction

Hierarchical clustering is a method of cluster analysis that builds a hierarchy of clusters. Unlike K-Means, it doesn't require specifying the number of clusters beforehand and provides a dendrogram for visualization.

### Types of Hierarchical Clustering:
- **Agglomerative (Bottom-up):** Start with each point as a cluster, merge closest pairs
- **Divisive (Top-down):** Start with all points in one cluster, recursively split

In [ ]:
# Install required packages
!pip install numpy pandas matplotlib seaborn scikit-learn scipy plotly -q

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, load_iris, load_wine, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster, cophenet
from scipy.spatial.distance import pdist
import plotly.express as px
import plotly.figure_factory as ff
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

print("All libraries imported successfully!")

<a id='theory'></a>
## 2. Theory and Concepts

### Linkage Methods:

| Method | Description | Formula |
|--------|-------------|--------|
| **Single** | Minimum distance between clusters | $d(A,B) = \min_{a \in A, b \in B} d(a,b)$ |
| **Complete** | Maximum distance between clusters | $d(A,B) = \max_{a \in A, b \in B} d(a,b)$ |
| **Average** | Average distance between all pairs | $d(A,B) = \frac{1}{|A||B|} \sum_{a \in A} \sum_{b \in B} d(a,b)$ |
| **Ward** | Minimizes variance within clusters | Minimizes total within-cluster variance |

### Distance Metrics:
- **Euclidean:** $\sqrt{\sum_i (x_i - y_i)^2}$
- **Manhattan:** $\sum_i |x_i - y_i|$
- **Cosine:** $1 - \frac{x \cdot y}{||x|| \cdot ||y||}$

<a id='dataset'></a>
## 3. Dataset Preparation

In [ ]:
# Generate synthetic datasets
# Dataset 1: Well-separated blobs
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, cluster_std=0.6, random_state=42)

# Dataset 2: Moon-shaped clusters (non-convex)
X_moons, y_moons = make_moons(n_samples=300, noise=0.05, random_state=42)

# Dataset 3: Iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Dataset 4: Wine dataset
wine = load_wine()
X_wine = wine.data
y_wine = wine.target

print("Datasets loaded:")
print(f"  - Blobs: {X_blobs.shape}")
print(f"  - Moons: {X_moons.shape}")
print(f"  - Iris: {X_iris.shape}")
print(f"  - Wine: {X_wine.shape}")

In [ ]:
# Visualize synthetic datasets
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
scatter1 = ax1.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='viridis', alpha=0.7, s=50)
ax1.set_title('Blob Dataset (4 clusters)', fontsize=14)
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')
plt.colorbar(scatter1, ax=ax1, label='Cluster')

ax2 = axes[1]
scatter2 = ax2.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='viridis', alpha=0.7, s=50)
ax2.set_title('Moon Dataset (2 clusters)', fontsize=14)
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')
plt.colorbar(scatter2, ax=ax2, label='Cluster')

plt.tight_layout()
plt.show()

In [ ]:
# Standardize data
scaler = StandardScaler()
X_blobs_scaled = scaler.fit_transform(X_blobs)
X_moons_scaled = scaler.fit_transform(X_moons)
X_iris_scaled = scaler.fit_transform(X_iris)
X_wine_scaled = scaler.fit_transform(X_wine)

print("Data standardized successfully!")

<a id='agglomerative'></a>
## 4. Agglomerative Clustering

In [ ]:
# Apply Agglomerative Clustering with Ward linkage
agg_ward = AgglomerativeClustering(n_clusters=4, linkage='ward')
labels_ward = agg_ward.fit_predict(X_blobs_scaled)

print(f"Number of clusters: {len(np.unique(labels_ward))}")
print(f"Cluster distribution: {np.bincount(labels_ward)}")

In [ ]:
# Visualize clustering results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
scatter1 = ax1.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='viridis', alpha=0.7, s=50)
ax1.set_title('True Labels', fontsize=14)
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')
plt.colorbar(scatter1, ax=ax1, label='Cluster')

ax2 = axes[1]
scatter2 = ax2.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels_ward, cmap='viridis', alpha=0.7, s=50)
ax2.set_title('Hierarchical Clustering (Ward)', fontsize=14)
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')
plt.colorbar(scatter2, ax=ax2, label='Cluster')

plt.tight_layout()
plt.show()

<a id='dendrogram'></a>
## 5. Dendrogram Analysis

A dendrogram is a tree-like diagram that shows the hierarchical relationship between clusters.

In [ ]:
# Create linkage matrix
linkage_matrix = linkage(X_blobs_scaled, method='ward')

# Plot dendrogram
plt.figure(figsize=(16, 8))
dendrogram(linkage_matrix, 
           truncate_mode='lastp',
           p=30,
           leaf_rotation=90,
           leaf_font_size=10,
           show_contracted=True)
plt.title('Hierarchical Clustering Dendrogram (Ward Linkage)', fontsize=16)
plt.xlabel('Sample Index or (Cluster Size)', fontsize=12)
plt.ylabel('Distance', fontsize=12)
plt.axhline(y=15, color='r', linestyle='--', label='Cut at distance=15 (4 clusters)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Full dendrogram for smaller subset
# Take a subset for clearer visualization
subset_idx = np.random.choice(len(X_blobs_scaled), 50, replace=False)
X_subset = X_blobs_scaled[subset_idx]
y_subset = y_blobs[subset_idx]

linkage_subset = linkage(X_subset, method='ward')

plt.figure(figsize=(16, 10))
dendrogram(linkage_subset,
           leaf_rotation=90,
           leaf_font_size=8,
           labels=y_subset)
plt.title('Full Dendrogram (50 samples subset)', fontsize=16)
plt.xlabel('Sample (True Label)', fontsize=12)
plt.ylabel('Distance', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Cophenetic correlation coefficient
# Measures how faithfully the dendrogram preserves pairwise distances
c, coph_dists = cophenet(linkage_matrix, pdist(X_blobs_scaled))
print(f"Cophenetic Correlation Coefficient: {c:.4f}")
print("(Values close to 1 indicate the dendrogram preserves original distances well)")

In [ ]:
# Determine optimal number of clusters from dendrogram
def plot_dendrogram_with_cuts(X, method='ward', max_clusters=10):
    """Plot dendrogram with different cut levels"""
    linkage_mat = linkage(X, method=method)
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    for idx, n_clusters in enumerate([2, 3, 4, 5]):
        ax = axes[idx // 2, idx % 2]
        
        # Get cluster labels
        labels = fcluster(linkage_mat, n_clusters, criterion='maxclust')
        
        # Calculate silhouette score
        sil_score = silhouette_score(X, labels)
        
        # Plot
        dendrogram(linkage_mat, ax=ax, truncate_mode='lastp', p=20,
                   leaf_rotation=90, leaf_font_size=8)
        ax.set_title(f'{n_clusters} Clusters (Silhouette: {sil_score:.3f})', fontsize=12)
        ax.set_xlabel('Sample Index')
        ax.set_ylabel('Distance')
    
    plt.suptitle('Dendrogram with Different Cluster Cuts', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

plot_dendrogram_with_cuts(X_blobs_scaled)

<a id='linkage'></a>
## 6. Different Linkage Methods Comparison

In [ ]:
# Compare different linkage methods
linkage_methods = ['single', 'complete', 'average', 'ward']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for idx, method in enumerate(linkage_methods):
    # Fit clustering
    if method == 'ward':
        agg = AgglomerativeClustering(n_clusters=4, linkage=method)
    else:
        agg = AgglomerativeClustering(n_clusters=4, linkage=method)
    labels = agg.fit_predict(X_blobs_scaled)
    
    # Scatter plot
    ax1 = axes[0, idx]
    scatter = ax1.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap='viridis', alpha=0.7, s=40)
    ax1.set_title(f'{method.capitalize()} Linkage', fontsize=14)
    ax1.set_xlabel('Feature 1')
    ax1.set_ylabel('Feature 2')
    
    # Dendrogram
    ax2 = axes[1, idx]
    linkage_mat = linkage(X_blobs_scaled, method=method)
    dendrogram(linkage_mat, ax=ax2, truncate_mode='lastp', p=15,
               leaf_rotation=90, leaf_font_size=8, no_labels=True)
    ax2.set_title(f'Dendrogram ({method})', fontsize=12)
    ax2.set_xlabel('Sample')
    ax2.set_ylabel('Distance')

plt.suptitle('Comparison of Linkage Methods', fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Compare linkage methods on moon dataset (non-convex clusters)
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for idx, method in enumerate(linkage_methods):
    agg = AgglomerativeClustering(n_clusters=2, linkage=method)
    labels = agg.fit_predict(X_moons_scaled)
    
    ax = axes[idx]
    scatter = ax.scatter(X_moons[:, 0], X_moons[:, 1], c=labels, cmap='viridis', alpha=0.7, s=50)
    
    # Calculate metrics
    sil = silhouette_score(X_moons_scaled, labels)
    ari = adjusted_rand_score(y_moons, labels)
    
    ax.set_title(f'{method.capitalize()}\nSilhouette: {sil:.3f}, ARI: {ari:.3f}', fontsize=12)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.suptitle('Linkage Methods on Moon Dataset', fontsize=16, y=1.05)
plt.tight_layout()
plt.show()

print("\nNote: Single linkage works best for non-convex (moon-shaped) clusters!")

In [ ]:
# Quantitative comparison of linkage methods
print("=" * 70)
print("LINKAGE METHODS COMPARISON")
print("=" * 70)

results = []

for method in linkage_methods:
    # Blobs dataset
    agg_blobs = AgglomerativeClustering(n_clusters=4, linkage=method)
    labels_blobs = agg_blobs.fit_predict(X_blobs_scaled)
    
    # Moons dataset
    agg_moons = AgglomerativeClustering(n_clusters=2, linkage=method)
    labels_moons = agg_moons.fit_predict(X_moons_scaled)
    
    # Cophenetic correlation
    linkage_mat = linkage(X_blobs_scaled, method=method)
    coph_corr, _ = cophenet(linkage_mat, pdist(X_blobs_scaled))
    
    results.append({
        'Linkage': method.capitalize(),
        'Blobs Silhouette': silhouette_score(X_blobs_scaled, labels_blobs),
        'Blobs ARI': adjusted_rand_score(y_blobs, labels_blobs),
        'Moons Silhouette': silhouette_score(X_moons_scaled, labels_moons),
        'Moons ARI': adjusted_rand_score(y_moons, labels_moons),
        'Cophenetic Corr': coph_corr
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

<a id='metrics'></a>
## 7. Clustering Quality Metrics

In [ ]:
def comprehensive_evaluation(X, labels_pred, labels_true=None, name="Dataset"):
    """Comprehensive clustering evaluation"""
    print(f"\n{'='*60}")
    print(f"CLUSTERING EVALUATION: {name}")
    print(f"{'='*60}")
    
    # Internal metrics
    print("\n--- Internal Metrics (No ground truth needed) ---")
    print(f"Silhouette Score:        {silhouette_score(X, labels_pred):.4f}")
    print(f"Calinski-Harabasz Index: {calinski_harabasz_score(X, labels_pred):.4f}")
    print(f"Davies-Bouldin Index:    {davies_bouldin_score(X, labels_pred):.4f}")
    
    # External metrics
    if labels_true is not None:
        print("\n--- External Metrics (Ground truth required) ---")
        print(f"Adjusted Rand Index:     {adjusted_rand_score(labels_true, labels_pred):.4f}")
        print(f"Normalized Mutual Info:  {normalized_mutual_info_score(labels_true, labels_pred):.4f}")
    
    # Cluster statistics
    print("\n--- Cluster Statistics ---")
    unique, counts = np.unique(labels_pred, return_counts=True)
    for cluster, count in zip(unique, counts):
        print(f"Cluster {cluster}: {count} samples ({100*count/len(labels_pred):.1f}%)")

# Evaluate on Iris dataset
agg_iris = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_iris_pred = agg_iris.fit_predict(X_iris_scaled)
comprehensive_evaluation(X_iris_scaled, labels_iris_pred, y_iris, "Iris Dataset")

In [ ]:
# Evaluate on Wine dataset
agg_wine = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_wine_pred = agg_wine.fit_predict(X_wine_scaled)
comprehensive_evaluation(X_wine_scaled, labels_wine_pred, y_wine, "Wine Dataset")

In [ ]:
# Optimal number of clusters analysis
def find_optimal_clusters(X, max_clusters=10, linkage='ward'):
    """Find optimal number of clusters using various metrics"""
    silhouette_scores = []
    calinski_scores = []
    davies_scores = []
    
    k_range = range(2, max_clusters + 1)
    
    for k in k_range:
        agg = AgglomerativeClustering(n_clusters=k, linkage=linkage)
        labels = agg.fit_predict(X)
        
        silhouette_scores.append(silhouette_score(X, labels))
        calinski_scores.append(calinski_harabasz_score(X, labels))
        davies_scores.append(davies_bouldin_score(X, labels))
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    ax1 = axes[0]
    ax1.plot(k_range, silhouette_scores, 'bo-', linewidth=2, markersize=8)
    ax1.set_xlabel('Number of Clusters', fontsize=12)
    ax1.set_ylabel('Silhouette Score', fontsize=12)
    ax1.set_title('Silhouette Score (Higher is better)', fontsize=14)
    ax1.grid(True, alpha=0.3)
    best_k_sil = k_range[np.argmax(silhouette_scores)]
    ax1.axvline(x=best_k_sil, color='r', linestyle='--', label=f'Best K={best_k_sil}')
    ax1.legend()
    
    ax2 = axes[1]
    ax2.plot(k_range, calinski_scores, 'go-', linewidth=2, markersize=8)
    ax2.set_xlabel('Number of Clusters', fontsize=12)
    ax2.set_ylabel('Calinski-Harabasz Index', fontsize=12)
    ax2.set_title('Calinski-Harabasz Index (Higher is better)', fontsize=14)
    ax2.grid(True, alpha=0.3)
    best_k_ch = k_range[np.argmax(calinski_scores)]
    ax2.axvline(x=best_k_ch, color='r', linestyle='--', label=f'Best K={best_k_ch}')
    ax2.legend()
    
    ax3 = axes[2]
    ax3.plot(k_range, davies_scores, 'ro-', linewidth=2, markersize=8)
    ax3.set_xlabel('Number of Clusters', fontsize=12)
    ax3.set_ylabel('Davies-Bouldin Index', fontsize=12)
    ax3.set_title('Davies-Bouldin Index (Lower is better)', fontsize=14)
    ax3.grid(True, alpha=0.3)
    best_k_db = k_range[np.argmin(davies_scores)]
    ax3.axvline(x=best_k_db, color='g', linestyle='--', label=f'Best K={best_k_db}')
    ax3.legend()
    
    plt.suptitle('Optimal Number of Clusters Analysis', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()
    
    return best_k_sil, best_k_ch, best_k_db

best_sil, best_ch, best_db = find_optimal_clusters(X_iris_scaled, max_clusters=10)
print(f"\nOptimal K suggestions:")
print(f"  - By Silhouette Score: {best_sil}")
print(f"  - By Calinski-Harabasz: {best_ch}")
print(f"  - By Davies-Bouldin: {best_db}")

<a id='realworld'></a>
## 8. Real-World Application: Customer Segmentation

In [ ]:
# Create synthetic customer data
np.random.seed(42)
n_customers = 500

# Generate customer features
customer_data = {
    'Age': np.concatenate([
        np.random.normal(25, 5, 150),  # Young
        np.random.normal(45, 8, 200),  # Middle-aged
        np.random.normal(65, 7, 150)   # Senior
    ]),
    'Annual_Income': np.concatenate([
        np.random.normal(30000, 8000, 150),
        np.random.normal(75000, 15000, 200),
        np.random.normal(50000, 12000, 150)
    ]),
    'Spending_Score': np.concatenate([
        np.random.normal(70, 15, 150),
        np.random.normal(50, 20, 200),
        np.random.normal(30, 10, 150)
    ]),
    'Purchase_Frequency': np.concatenate([
        np.random.normal(15, 5, 150),
        np.random.normal(8, 3, 200),
        np.random.normal(4, 2, 150)
    ])
}

df_customers = pd.DataFrame(customer_data)
df_customers = df_customers.clip(lower=0)  # Ensure no negative values

print("Customer Dataset:")
print(df_customers.describe())

In [ ]:
# Standardize customer data
X_customers = df_customers.values
X_customers_scaled = scaler.fit_transform(X_customers)

# Apply hierarchical clustering
agg_customers = AgglomerativeClustering(n_clusters=4, linkage='ward')
customer_labels = agg_customers.fit_predict(X_customers_scaled)

df_customers['Cluster'] = customer_labels

print("\nCluster Distribution:")
print(df_customers['Cluster'].value_counts().sort_index())

In [ ]:
# Visualize customer segments
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Age vs Income
ax1 = axes[0, 0]
scatter1 = ax1.scatter(df_customers['Age'], df_customers['Annual_Income'], 
                       c=customer_labels, cmap='viridis', alpha=0.6, s=50)
ax1.set_xlabel('Age', fontsize=12)
ax1.set_ylabel('Annual Income', fontsize=12)
ax1.set_title('Age vs Annual Income', fontsize=14)
plt.colorbar(scatter1, ax=ax1, label='Cluster')

# Income vs Spending
ax2 = axes[0, 1]
scatter2 = ax2.scatter(df_customers['Annual_Income'], df_customers['Spending_Score'], 
                       c=customer_labels, cmap='viridis', alpha=0.6, s=50)
ax2.set_xlabel('Annual Income', fontsize=12)
ax2.set_ylabel('Spending Score', fontsize=12)
ax2.set_title('Income vs Spending Score', fontsize=14)
plt.colorbar(scatter2, ax=ax2, label='Cluster')

# Age vs Spending
ax3 = axes[1, 0]
scatter3 = ax3.scatter(df_customers['Age'], df_customers['Spending_Score'], 
                       c=customer_labels, cmap='viridis', alpha=0.6, s=50)
ax3.set_xlabel('Age', fontsize=12)
ax3.set_ylabel('Spending Score', fontsize=12)
ax3.set_title('Age vs Spending Score', fontsize=14)
plt.colorbar(scatter3, ax=ax3, label='Cluster')

# Purchase Frequency vs Spending
ax4 = axes[1, 1]
scatter4 = ax4.scatter(df_customers['Purchase_Frequency'], df_customers['Spending_Score'], 
                       c=customer_labels, cmap='viridis', alpha=0.6, s=50)
ax4.set_xlabel('Purchase Frequency', fontsize=12)
ax4.set_ylabel('Spending Score', fontsize=12)
ax4.set_title('Purchase Frequency vs Spending Score', fontsize=14)
plt.colorbar(scatter4, ax=ax4, label='Cluster')

plt.suptitle('Customer Segmentation Analysis', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cluster profiles
print("\n" + "=" * 70)
print("CUSTOMER SEGMENT PROFILES")
print("=" * 70)

cluster_profiles = df_customers.groupby('Cluster').agg({
    'Age': ['mean', 'std'],
    'Annual_Income': ['mean', 'std'],
    'Spending_Score': ['mean', 'std'],
    'Purchase_Frequency': ['mean', 'std']
}).round(2)

print(cluster_profiles)

# Segment descriptions
print("\n" + "-" * 70)
print("SEGMENT DESCRIPTIONS")
print("-" * 70)

for cluster in range(4):
    cluster_data = df_customers[df_customers['Cluster'] == cluster]
    print(f"\nCluster {cluster} ({len(cluster_data)} customers):")
    print(f"  - Average Age: {cluster_data['Age'].mean():.1f} years")
    print(f"  - Average Income: ${cluster_data['Annual_Income'].mean():,.0f}")
    print(f"  - Average Spending Score: {cluster_data['Spending_Score'].mean():.1f}")
    print(f"  - Average Purchase Frequency: {cluster_data['Purchase_Frequency'].mean():.1f}/month")

In [ ]:
# Dendrogram for customer data
linkage_customers = linkage(X_customers_scaled, method='ward')

plt.figure(figsize=(16, 8))
dendrogram(linkage_customers, truncate_mode='lastp', p=40,
           leaf_rotation=90, leaf_font_size=8)
plt.title('Customer Segmentation Dendrogram', fontsize=16)
plt.xlabel('Customer Index', fontsize=12)
plt.ylabel('Distance', fontsize=12)
plt.axhline(y=10, color='r', linestyle='--', label='Cut for 4 clusters')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of cluster characteristics
cluster_means = df_customers.groupby('Cluster')[['Age', 'Annual_Income', 'Spending_Score', 'Purchase_Frequency']].mean()

# Normalize for heatmap
cluster_means_normalized = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())

plt.figure(figsize=(10, 6))
sns.heatmap(cluster_means_normalized, annot=cluster_means.round(1), 
            cmap='YlOrRd', fmt='', linewidths=0.5)
plt.title('Customer Segment Characteristics Heatmap', fontsize=14)
plt.xlabel('Features', fontsize=12)
plt.ylabel('Cluster', fontsize=12)
plt.tight_layout()
plt.show()

<a id='conclusion'></a>
## 9. Conclusion

### Summary

In this notebook, we explored **Hierarchical Clustering** using scikit-learn and scipy.

### Key Findings:

1. **Linkage Methods:**
   - **Ward:** Best for compact, spherical clusters
   - **Single:** Best for elongated or non-convex clusters
   - **Complete:** Produces compact clusters but sensitive to outliers
   - **Average:** Good balance between single and complete

2. **Advantages of Hierarchical Clustering:**
   - No need to specify K beforehand
   - Dendrogram provides interpretable visualization
   - Can capture hierarchical relationships

3. **Limitations:**
   - Computationally expensive for large datasets O(n²)
   - Cannot undo merges (agglomerative)
   - Sensitive to noise and outliers

4. **Quality Metrics:**
   - Silhouette Score for cluster separation
   - Cophenetic Correlation for dendrogram quality
   - Davies-Bouldin Index for cluster compactness

### References:
- Müllner, D. (2011). "Modern hierarchical, agglomerative clustering algorithms"
- Ward, J. H. (1963). "Hierarchical Grouping to Optimize an Objective Function"

In [ ]:
# Final summary
print("=" * 70)
print("           HIERARCHICAL CLUSTERING - FINAL SUMMARY")
print("=" * 70)
print("\n✓ Implemented Agglomerative Clustering using scikit-learn")
print("✓ Analyzed dendrograms for cluster visualization")
print("✓ Compared 4 linkage methods: single, complete, average, ward")
print("✓ Tested on synthetic data (blobs, moons)")
print("✓ Applied to real datasets (Iris, Wine)")
print("✓ Demonstrated customer segmentation use case")
print("✓ Evaluated using multiple clustering quality metrics")
print("✓ Analyzed optimal number of clusters")
print("\n" + "=" * 70)